This script is meant to contain the periodic operations when continuing working on the ec2 instance day by day

In [ ]:
import boto3
import time

ec2 = boto3.client('ec2')

Get the needed instance id by the human readable name

In [ ]:
response = ec2.describe_instances()

instance_name = "mlops-mhofmann"
instance_id = None

for resp in response['Reservations']:
    resp = resp["Instances"][0]
    tags = resp.get("Tags", [])
    # print(tags)

    for tag in tags:
        if tag.get("Key") == "Name" and tag.get("Value") == instance_name:
            instance_id = resp["InstanceId"]
            print(f"Found instance {instance_name} with ID {instance_id}")

if not instance_id:
    print(f"Instance {instance_name} not found.")


Starting and stopping the instance to save money

In [ ]:
def wait_for_status(instance_id, target_status):
    while True:
        response = ec2.describe_instances(InstanceIds=[instance_id])
        status = response['Reservations'][0]['Instances'][0]['State']['Name']
        if status == target_status:
            print(f"Instance {instance_id} is now in '{target_status}' state.")
            break
        else:
            print(f"Instance {instance_id} is still in status '{status}'. Waiting for status '{target_status}'...")
        time.sleep(10)  # check every 10 seconds

def start_instance(instance_id):
    ec2.start_instances(InstanceIds=[instance_id])
    print(f"Starting instance {instance_id}...")
    wait_for_status(instance_id, 'running')

def stop_instance(instance_id):
    ec2.stop_instances(InstanceIds=[instance_id])
    print(f"Stopping instance {instance_id}...")
    wait_for_status(instance_id, 'stopped')

Starting the instance

In [ ]:
start_instance(instance_id)

Stopping the instance

In [ ]:
stop_instance(instance_id)

Get the public ip of the server to access things like api etc later.\
The public ip always changes after starting and stopping the instance.

In [ ]:
response = ec2.describe_instances(InstanceIds=[instance_id])
public_ip = response['Reservations'][0]['Instances'][0]['PublicIpAddress']
print(f"Public IP of instance {instance_id} is {public_ip}")